# 02 - Data Preprocessing

> Research prototype only. The dataset is simulated and labels are experimental proxies. No result here validates field deployment or early-warning lead time.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_config, load_raw_data
from src.data_validation import validate_target
from src.feature_engineering import add_engineered_features, select_feature_columns, verify_flow_conventions

config = load_config()
frame = load_raw_data(config=config)
target = validate_target(frame, config)

## Cleaning decisions

There are no missing values in the raw file; five exact duplicate rows are retained because they may represent genuine steady-state simulator samples. Constant columns and target-derived columns are excluded from model inputs.

In [2]:
print('Missing values:', int(frame.isna().sum().sum()))
print('Duplicate rows:', frame.duplicated().sum())
featured = add_engineered_features(frame, config)
print(verify_flow_conventions(featured))
columns = select_feature_columns(featured, config)
print('Selected features:', columns)

Missing values: 0
Duplicate rows: 5
{'verified': True, 'max_abs_residual': 1.09999999997612e-06, 'interpretation': 'MRFlow = FIn - FOut; therefore Flow_Imbalance = FOut - FIn = -MRFlow. Positive values mean return flow exceeds input flow (pit gain).'}
Selected features: ['CircFlow', 'FRate', 'FDensity', 'SMSpeed', 'MVis', 'DPPress', 'FIn', 'FOut', 'RoPen', 'WoBit', 'HLoad', 'WBoPress', 'ATMPV', 'ATMYP', 'Flow_Imbalance', 'Flow_Imbalance_Ratio']


## Scaling and imputation

The `ColumnTransformer` below is fitted on training data only. Robust scaling is used because genuine transient kick responses should not be treated as errors.

In [3]:
from src.preprocessing import build_preprocessor, split_data, make_validation_split
split = split_data(featured[columns], target.labels, config)
preprocessor = build_preprocessor(columns, config)
X_train_t = preprocessor.fit_transform(split.X_train)
print('Transformed training matrix:', X_train_t.shape)
fit, val, y_fit, y_val = make_validation_split(split.X_train, split.y_train, split.train_groups, config)
print('Fit/validation rows:', len(fit), len(val))

Transformed training matrix: (1200, 16)
Fit/validation rows: 950 250


## Group-aware split

Contiguous 25-row blocks remain intact and are stratified by the proxy class, reducing leakage from adjacent near-duplicate measurements. The test set is not used for any fitting or tuning.

In [4]:
print('Train/test rows:', len(split.X_train), len(split.X_test))
print('Positive fraction:', split.y_train.mean(), split.y_test.mean())
split.X_train.assign(target=split.y_train.to_numpy()).head()

Train/test rows: 1200 475
Positive fraction: 0.395 0.3684210526315789


,CircFlow,FRate,FDensity,SMSpeed,MVis,DPPress,FIn,FOut,RoPen,WoBit,HLoad,WBoPress,ATMPV,ATMYP,Flow_Imbalance,Flow_Imbalance_Ratio,target
0,0.000000,0.000000,0.00000,0.000000,0.00000,1730.271,0.952940,0.952891,20.44352,39773.86,310828.4,7001.946,10.00008,20.00007,-0.000049,-0.000051,0
1,0.963155,-0.980443,78.54707,-0.005669,10.00045,1729.887,0.953122,0.980443,20.40985,39722.29,310839.9,6957.020,10.00008,20.00007,0.027321,0.028665,0
2,0.963571,-0.982254,78.54707,-0.005670,10.00045,1729.808,0.953133,0.982254,20.41020,39726.00,310833.7,6957.143,10.00008,20.00007,0.029121,0.030553,0
3,0.964249,-0.985531,78.54707,-0.005670,10.00045,1729.638,0.953151,0.985531,20.41112,39733.50,310821.8,6957.366,10.00008,20.00007,0.032380,0.033972,0
4,0.964524,-0.987007,78.54707,-0.005670,10.00045,1729.547,0.953158,0.987007,20.41166,39737.29,310816.0,6957.467,10.00008,20.00007,0.033849,0.035512,0
